In [5]:
import pandas as pd
import pygmt
from pyproj import Transformer

# 1. Load the input data
input_file = "Final_Tenerife_Gravity_Disturbance_All_Datastes.csv"
df = pd.read_csv(input_file)

# 2. Get the geographic bounding box to load the geoid model
region = [
    df["longitude"].min() - 0.1,
    df["longitude"].max() + 0.1,
    df["latitude"].min() - 0.1,
    df["latitude"].max() + 0.1,
]

# 3. Load the EGM2008 geoid model via PyGMT
geoid = pygmt.datasets.load_earth_geoid(region=region, resolution="01m")

# Track coordinates to sample the geoid at our data points
track_df = df[["longitude", "latitude"]]
sampled_geoid = pygmt.grdtrack(points=track_df, grid=geoid, newcolname="N")
df["N"] = sampled_geoid["N"]

# 4. Convert Ellipsoidal height (h) to Geoidal height / Sea Level (H)
# Formula: H = h - N
df["elevation_geoidal"] = df["height"] - df["N"]

# 5. Convert Gravity Unit from mGal to uGal
# 1 mGal = 1000 uGal
df["G_ref_uGal"] = df["G_ref"] * 1000

# 6. Convert WGS84 Lat/Lon to UTM Coordinates (Zone 28N for Tenerife)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32628", always_xy=True)
df["easting"], df["northing"] = transformer.transform(
    df["longitude"].values, df["latitude"].values
)

# 7. Select the final output columns (now including the converted gravity)
output_df = df[["easting", "northing", "elevation_geoidal", "G_ref_uGal"]]

# 8. Save to files with exactly 2 decimal places and no column titles
output_file_txt = "Transformed_Tenerife_Gravity_Data.txt"
output_file_csv = "Transformed_Tenerife_Gravity_Data.csv"

# Save as space-separated text file
output_df.to_csv(output_file_txt, sep=" ", header=False, index=False, float_format="%.2f")

# Save as comma-separated CSV file
output_df.to_csv(output_file_csv, header=False, index=False, float_format="%.2f")

print(f"Successfully saved files with 2 decimal precision and uGal conversion:\n - {output_file_txt}\n - {output_file_csv}")

Successfully saved files with 2 decimal precision and uGal conversion:
 - Transformed_Tenerife_Gravity_Data.txt
 - Transformed_Tenerife_Gravity_Data.csv
